# 12 — Capstone: Explanation Audit

End-to-end XAI pipeline combining SHAP, integrated gradients, counterfactuals, stability testing, and compliance reporting.

In [ ]:
# Setup: train the model
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from datetime import datetime

np.random.seed(42)
torch.manual_seed(42)

feature_names = ['income', 'credit_score', 'debt_ratio',
                 'employment_yrs', 'loan_amount', 'credit_lines',
                 'payment_history', 'utilisation']

X, y = make_classification(n_samples=2000, n_features=8, n_informative=5,
                            n_redundant=2, random_state=42)
scaler = StandardScaler()
X = scaler.fit_transform(X)

class CreditNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

Xtr = torch.tensor(X[:1600], dtype=torch.float32)
ytr = torch.tensor(y[:1600], dtype=torch.float32)
Xte = torch.tensor(X[1600:], dtype=torch.float32)

model = CreditNet()
opt = torch.optim.Adam(model.parameters(), lr=5e-3)
for _ in range(150):
    loss = nn.BCELoss()(model(Xtr), ytr)
    opt.zero_grad(); loss.backward(); opt.step()

model.eval()
with torch.no_grad():
    acc = ((model(Xte) > 0.5).float() == torch.tensor(y[1600:], dtype=torch.float32)).float().mean()
print(f'Model accuracy: {acc:.3f}')

In [ ]:
# Component 1: SHAP (KernelSHAP approximation)
def kernel_shap_fast(model_fn, x, X_background, n_samples=512):
    n_features = x.shape[0]
    background_mean = X_background.mean(axis=0)
    base_val = model_fn(background_mean.reshape(1, -1))[0]
    masks = np.random.randint(0, 2, size=(n_samples, n_features))
    masks[0] = 1; masks[1] = 0
    X_masked = np.where(masks, x, background_mean)
    preds = model_fn(X_masked)
    z = masks.sum(axis=1).clip(1, n_features - 1)
    from math import comb
    weights = np.array([(n_features-1)/(comb(n_features,int(zi))*int(zi)*(n_features-int(zi))) for zi in z])
    W = np.diag(weights)
    from numpy.linalg import lstsq
    phi, _, _, _ = lstsq(masks.T @ W @ masks, masks.T @ W @ (preds - base_val), rcond=None)
    return phi, base_val

def model_fn(X_np):
    with torch.no_grad():
        return model(torch.tensor(X_np, dtype=torch.float32)).numpy()

x_sample = X[1600]  # first test sample
shap_vals, base = kernel_shap_fast(model_fn, x_sample, X[:1600])
print('SHAP attributions:')
for i, v in enumerate(shap_vals):
    print(f'  {feature_names[i]}: {v:+.4f}')
print(f'Efficiency check: {shap_vals.sum() + base:.4f} ≈ {model_fn(x_sample.reshape(1,-1))[0]:.4f}')

In [ ]:
# Component 2: Integrated Gradients
def ig(model, x_t, n_steps=50):
    baseline = torch.zeros_like(x_t)
    alphas = torch.linspace(0, 1, n_steps + 1)
    interp = baseline + alphas.unsqueeze(1) * (x_t - baseline)
    interp.requires_grad_(True)
    preds = model(interp)
    grads = []
    for i in range(len(alphas)):
        if interp.grad is not None: interp.grad.zero_()
        preds[i].backward(retain_graph=True)
        grads.append(interp.grad[i].clone())
    grads = torch.stack(grads)
    return ((x_t - baseline) * grads[:-1].mean(0)).detach().numpy()

x_t = torch.tensor(x_sample, dtype=torch.float32)
ig_vals = ig(model, x_t)
print('IG attributions:')
for i, v in enumerate(ig_vals):
    print(f'  {feature_names[i]}: {v:+.4f}')

In [ ]:
# Component 3: Counterfactual recourse
def find_counterfactual(model, x, n_steps=300, lr=0.1):
    x_t = torch.tensor(x, dtype=torch.float32)
    cf = x_t.clone().unsqueeze(0).requires_grad_(True)
    opt_cf = torch.optim.Adam([cf], lr=lr)
    for _ in range(n_steps):
        pred = model(cf)
        loss = nn.BCELoss()(pred, torch.ones(1)) + 0.5 * ((cf - x_t)**2).mean()
        opt_cf.zero_grad(); loss.backward(); opt_cf.step()
    return cf.detach().squeeze().numpy()

with torch.no_grad():
    pred_score = model(x_t.unsqueeze(0)).item()
print(f'Original prediction: {pred_score:.3f} ({"DECLINED" if pred_score < 0.5 else "APPROVED"})')

cf = find_counterfactual(model, x_sample)
with torch.no_grad():
    cf_score = model(torch.tensor(cf, dtype=torch.float32).unsqueeze(0)).item()
print(f'Counterfactual prediction: {cf_score:.3f} ({"DECLINED" if cf_score < 0.5 else "APPROVED"})')

# Component 4: Stability (run SHAP 5 times)
runs = [kernel_shap_fast(model_fn, x_sample, X[:1600])[0] for _ in range(5)]
shap_std = np.std(runs, axis=0)
print(f'\nSHAP stability (std across 5 runs): {shap_std.mean():.4f}')

In [ ]:
# Final: visualise audit dashboard
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# SHAP
sorted_idx = np.argsort(np.abs(shap_vals))[::-1]
ax = axes[0, 0]
colors = ['steelblue' if v > 0 else 'tomato' for v in shap_vals[sorted_idx]]
ax.barh([feature_names[i] for i in sorted_idx], shap_vals[sorted_idx], color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('SHAP Attribution')

# IG
ig_sorted_idx = np.argsort(np.abs(ig_vals))[::-1]
ax = axes[0, 1]
colors_ig = ['steelblue' if v > 0 else 'tomato' for v in ig_vals[ig_sorted_idx]]
ax.barh([feature_names[i] for i in ig_sorted_idx], ig_vals[ig_sorted_idx], color=colors_ig)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Integrated Gradients')

# Counterfactual delta
ax = axes[1, 0]
delta = cf - x_sample
colors_cf = ['green' if v > 0 else 'red' for v in delta]
ax.barh(feature_names, delta, color=colors_cf)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Counterfactual Recourse')

# Stability
ax = axes[1, 1]
ax.bar(feature_names, shap_std, color='purple', alpha=0.7)
ax.set_title('SHAP Stability (std across 5 runs)')
ax.set_xticklabels(feature_names, rotation=45, ha='right')

plt.suptitle('XAI Audit Dashboard', fontsize=14)
plt.tight_layout()
plt.savefig('/tmp/xai_audit_dashboard.png', dpi=100, bbox_inches='tight')
plt.show()
print('XAI audit dashboard saved')
print('\nSeries 22 (XAI Deep Dive) complete.')